In [1]:
%pip install -q torch transformers accelerate bitsandbytes qwen-vl-utils matplotlib opencv-python pillow huggingface_hub

import json
import shutil
import subprocess
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
MODEL_DIR = Path("./models/qwen2_vl_2b_instruct")
IMAGE_DIR = Path("./data")
OUTPUT_FILE = "dataset.jsonl"
DATASET_REPO = "https://github.com/physicorym/designing_neural_network_architectures_2025_02.git"
DATASET_PATH = "seminar_05/data"

if not MODEL_DIR.is_dir() or not (MODEL_DIR / "config.json").exists():
    print("Downloading model from Hugging Face...")
    snapshot_download(MODEL_ID, local_dir=str(MODEL_DIR))

if not any(IMAGE_DIR.glob("*.jpg")):
    print("Downloading dataset from GitHub...")
    IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    tmp_repo = Path("_repo_tmp")
    if tmp_repo.exists():
        shutil.rmtree(tmp_repo)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", DATASET_REPO, str(tmp_repo)],
        check=True,
    )
    subprocess.run(["git", "sparse-checkout", "set", DATASET_PATH], cwd=tmp_repo, check=True)
    for image_file in (tmp_repo / DATASET_PATH).glob("*"):
        if image_file.is_file():
            shutil.copy2(image_file, IMAGE_DIR / image_file.name)
    shutil.rmtree(tmp_repo)
    print(f"Downloaded {len(list(IMAGE_DIR.glob('*.jpg')))} images")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    str(MODEL_DIR),
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=quantization_config,
    local_files_only=True,
)

processor = AutoProcessor.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
)

print("Model loaded from:", MODEL_DIR.resolve())
print("Images in dataset:", len(list(IMAGE_DIR.glob("*.jpg"))))


ModuleNotFoundError: No module named 'transformers'

In [ ]:
import matplotlib.pyplot as plt
import cv2

TEST_IMAGE = sorted(IMAGE_DIR.glob("*.jpg"))[0]

image = cv2.imread(str(TEST_IMAGE))[:, :, ::-1]
plt.imshow(image)
plt.show()

TEST_PROMPT = "Сколько людей в кадре? Опиши их очень подробно"

if not TEST_IMAGE.is_file():
    raise FileNotFoundError(f"Картинка не найдена: {TEST_IMAGE.resolve()}")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": str(TEST_IMAGE)},
            {"type": "text", "text": TEST_PROMPT},
        ],
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to(model.device)

generated_ids = model.generate(
    **inputs,
    max_new_tokens=128,
    do_sample=False,
)

generated_ids_trimmed = [
    out_ids[len(in_ids):]
    for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

answer = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0]

print("Промпт:", TEST_PROMPT)
print("Ответ:", answer)


In [ ]:
import os

PROMPT = """
Опиши изображение кратко и информативно.
Укажи:
- главные объекты
- действия
- сцену
Ответь одним предложением.
ОПИСЫВАЙ МАКСИМАЛЬНО ПОДРОБНО, КАЖДЫЙ ЭЛЕМЕНТ, КАЖДОЕ ДЕЙСТВИЕ!!!
"""

all_items = []

files = sorted(os.listdir(IMAGE_DIR))

for file_name in files[:5]:

    if not file_name.lower().endswith(
        (".jpg", ".jpeg", ".png", ".webp")
    ):
        continue

    image_path = str(IMAGE_DIR / file_name)

    print(f"Processing: {file_name}")

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_path
                },
                {
                    "type": "text",
                    "text": PROMPT
                }
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False
    )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(
            inputs.input_ids,
            generated_ids
        )
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    item = {
        "image": image_path,
        "text": output_text
    }

    all_items.append(item)

    print(output_text)


In [ ]:


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in all_items:
        f.write(
            json.dumps(item, ensure_ascii=False) + "\n"
        )

print(f"Saved to {OUTPUT_FILE}")
